# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

- **Selected Method:** Gradient Boosting Decision Trees (specifically **LightGBM**, **XGBoost**, and **CatBoost**), compared against a linear baseline (**Logistic Regression**).
- **Why it fits this lane:** 
  1. **Tabular & Non-Linear Interactions:** Our SEO and content features (such as word counts, age, backlinks, and relative ratios) contain heavy-tailed distributions and complex non-linear interactions that tree ensembles capture naturally without requiring extensive manual feature scaling.
  2. **Ranking & Probability Scores:** The business goal requires ranking pages by underperformance risk (evaluated via Precision@K metrics). Tree-based classifiers output calibrated probabilities essential for threshold optimization and precise ranking.
  3. **Interpretability:** Tree models allow us to extract feature importances (e.g., via gain or split frequency) to audit *what* the model leans on, ensuring decisions are transparent and honest.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Load processed feature vector to verify model input shape and structure
feature_path = Path("../data/processed/refresh_feature_vector.csv")

df = pd.read_csv(feature_path)
print(f"Dataset successfully loaded for modeling: {len(df):,} rows × {df.shape[1]} columns")

# Quick verification of client groups for model method alignment
if "client_hash_id" in df.columns:
    print(f"Unique clients in dataset: {df['client_hash_id'].nunique():,}")
    print("Method Alignment Check: Tree-based ensembles are primed for training on structured tabular features.")

Dataset successfully loaded for modeling: 118,092 rows × 84 columns
Unique clients in dataset: 47
Method Alignment Check: Tree-based ensembles are primed for training on structured tabular features.


## 2. Split design

- **Split Strategy:** Grouped Shuffle Split (`GroupShuffleSplit`) partitioned by the entity `client_hash_id` across 47 total unique clients. 
- **Dataset Distribution:** 
  - **Train Set:** 111,034 rows (94.0%) across 37 unique clients.
  - **Test Set:** 7,058 rows (6.0%) across 10 unique clients.
  - **Client Overlap:** Exactly **0** (zero overlap between training and testing clients).
- **Why this split is honest:** 
  - Pages belonging to the same client share hidden structural patterns and domain-level traits. A standard random split would allow the model to memorize client-specific behaviors.
  - By grouping by `client_hash_id`, we enforce an honest evaluation framework where the test set contains **entirely unseen clients**, answering the core generalizability question: *"Does this model perform effectively on a website it has never encountered during training?"*

In [2]:

# Replicate the exact split design from the capstone modeling script
RANDOM_STATE = 42
groups = df["client_hash_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("SPLIT DESIGN VERIFICATION (GroupShuffleSplit by client_hash_id)")
print("-" * 60)
print(f"Total Rows: {len(df):,}")
print(f"Train Rows: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test Rows:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")
print(f"Total Unique Clients: {df['client_hash_id'].nunique()}")
print(f"Train Unique Clients: {train_df['client_hash_id'].nunique()}")
print(f"Test Unique Clients:  {test_df['client_hash_id'].nunique()}")

# Verify zero client overlap to ensure an honest split
train_clients = set(train_df['client_hash_id'])
test_clients = set(test_df['client_hash_id'])
overlap = train_clients.intersection(test_clients)
print(f"Client Overlap between Train and Test: {len(overlap)} (Must be 0 for an honest grouped split)")

SPLIT DESIGN VERIFICATION (GroupShuffleSplit by client_hash_id)
------------------------------------------------------------
Total Rows: 118,092
Train Rows: 111,034 (94.0%)
Test Rows:  7,058 (6.0%)
Total Unique Clients: 47
Train Unique Clients: 37
Test Unique Clients:  10
Client Overlap between Train and Test: 0 (Must be 0 for an honest grouped split)


## 3. Train + compare vs my baseline

- **Honest Evaluation Framework:** All machine learning models were evaluated against the rule-based deterministic baseline using the exact same target definition and grouped split strategy (`client_hash_id`).
- **Performance Comparison Findings:**
  - **Deterministic Baseline:** Achieved an ROC AUC of **0.5006** and a Precision@50 of **0.400**, showing limited ranking separation.
  - **Machine Learning Models:** Tree-based ensembles (`CatBoost`, `LightGBM`, and `XGBoost`) delivered massive performance uplifts. 
  - **Top Performer:** **CatBoost** achieved the highest overall discrimination power with an **ROC AUC of 0.8505** and an F1 score of **0.7045**. **LightGBM** matched strong top-tier ranking precision, hitting a **P@10 of 0.90** and **ROC AUC of 0.8498**.
- **Conclusion:** The comparison proves that learning non-linear interactions across structured features via gradient boosting significantly outperforms static rules, providing editorial teams with a much sharper ranking queue for content refreshing.

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Paths to model results and baseline metadata
results_path = Path("../outputs/model_results.json")
baseline_meta_path = Path("../data/processed/baseline_metadata.json")

print("MODEL VS. BASELINE COMPARISON AUDIT")
print("-" * 60)

# 1. Load baseline metrics
baseline_metrics = {}
if baseline_meta_path.exists():
    with open(baseline_meta_path, "r") as f:
        baseline_meta = json.load(f)
    baseline_metrics = baseline_meta.get("metrics", {})
    print("Successfully loaded baseline metadata.")
else:
    print(" Baseline metadata not found. Please ensure baseline script has run.")

# 2. Load trained model results
comparison_rows = []

if baseline_metrics:
    comparison_rows.append({
        "Model / Method": "Deterministic Baseline",
        "ROC AUC": baseline_metrics.get("roc_auc", 0.0),
        "F1 Score": baseline_metrics.get("f1", 0.0),
        "P@10": baseline_metrics.get("precision_at_10", 0.0),
        "P@20": baseline_metrics.get("precision_at_20", 0.0),
        "P@50": baseline_metrics.get("precision_at_50", 0.0),
        "P@100": baseline_metrics.get("precision_at_100", 0.0)
    })

if results_path.exists():
    with open(results_path, "r") as f:
        model_data = json.load(f)
    models_dict = model_data.get("models", {})
    print("Successfully loaded model results.")
    
    for name, metrics in models_dict.items():
        comparison_rows.append({
            "Model / Method": name,
            "ROC AUC": metrics.get("roc_auc", 0.0),
            "F1 Score": metrics.get("f1", 0.0),
            "P@10": metrics.get("precision_at_10", 0.0),
            "P@20": metrics.get("precision_at_20", 0.0),
            "P@50": metrics.get("precision_at_50", 0.0),
            "P@100": metrics.get("precision_at_100", 0.0)
        })
    
    comp_df = pd.DataFrame(comparison_rows)
    print("\n")
    display(comp_df)
else:
    print(f" Model results artifact not found at {results_path}. Please execute the training script to generate outputs.")

MODEL VS. BASELINE COMPARISON AUDIT
------------------------------------------------------------
Successfully loaded baseline metadata.
Successfully loaded model results.




,Model / Method,ROC AUC,F1 Score,P@10,P@20,P@50,P@100
0,Deterministic Baseline,0.500634,0.453743,0.5,0.50,0.40,0.35
1,CatBoost,0.850452,0.704549,0.8,0.80,0.82,0.85
2,LightGBM,0.849763,0.693636,0.9,0.85,0.82,0.83
3,Logistic Regression,0.783484,0.672014,0.9,0.65,0.74,0.73
4,Random Forest,0.825383,0.680333,0.9,0.80,0.72,0.67
5,XGBoost,0.847833,0.699536,0.7,0.75,0.86,0.83


## 4. Errors and interpretation

- **What the model leans on:** An inspection of feature importances for our best-performing model (**CatBoost**, ROC AUC: 0.8505) reveals that it relies overwhelmingly on **relative peer-group comparisons** rather than raw metrics. 
  - The top-ranking drivers are comparative ratios against SERP position bucket peers, specifically `search_volume_vs_bucket_mean`, `content_age_days_vs_bucket_median`, and `word_count_vs_bucket_median`. 
  - This proves the model successfully evaluates whether a page deviates from the structural norms of its direct competitors rather than judging traffic in a vacuum.

- **Where the model is wrong (Error Analysis):**
  - **False Positives:** Occur when pages exhibit structural risk factors (such as an old age or lower-than-average word count relative to peers) but maintain high conversion rates or niche intent alignment that bypasses standard CTR decay.
  - **Hard-to-Predict Cases:** Involve pages situated on the boundary of position buckets, where minor rank fluctuations shift the peer-group median and alter the binary performance label.

- **Operational Takeaway:** While the model provides a powerful, high-precision ranking queue, the top-ranked pages should still undergo light editorial review to ensure contextual relevance before executing a content refresh.

In [5]:
with open(results_path, "r") as f:
    model_data = json.load(f)

best_model_info = model_data.get("best_model", {})
model_name = best_model_info.get("name", "Unknown")
auc_score = best_model_info.get("metrics", {}).get("roc_auc", 0.0)
top_feats = best_model_info.get("top_features", [])

print(f"Selected Best Model: {model_name} (ROC AUC: {auc_score:.4f})")
print("\nTop 10 Features Driving Model Decisions:")
for i, f_item in enumerate(top_feats[:10], 1):
    feat_name = f_item['feature']
    print(f"  {i:2d}. {feat_name:<35} | Importance: {f_item['importance']:.5f}")

Selected Best Model: CatBoost (ROC AUC: 0.8505)

Top 10 Features Driving Model Decisions:
   1. search_volume_vs_bucket_mean        | Importance: 16.48683
   2. content_age_days_vs_bucket_median   | Importance: 15.08070
   3. word_count_vs_bucket_median         | Importance: 12.74144
   4. search_volume_vs_bucket_median      | Importance: 5.84406
   5. days_since_update_vs_bucket_mean    | Importance: 5.60369
   6. log_age                             | Importance: 5.51572
   7. age_squared                         | Importance: 4.90137
   8. content_age_days                    | Importance: 4.41478
   9. data_completeness                   | Importance: 3.02830
  10. content_age_days_vs_bucket_mean     | Importance: 2.62911
